In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed/clean.parquet")

df = pd.read_parquet(DATA_PATH)
print("Rows:", len(df))
df.head()


Rows: 5000


,match_id,match_start_date,match_date,match_day,city,venue,bowler,age_years,spell_no,deliveries_match,...,fatigue_ewma,ck_delta_24h_upl,cmj_drop_pct,hrv_sim_ms,fatigue_score_0_100,injury_risk_score,injury_risk_label,fatigue_score_0_100_v3,injury_risk_prob_v3,injury_risk_label_v3
0,20210129_GalleInternational_Galle_AMFernando,2021-01-29,2021-01-31,3,Galle,Galle International Stadium,AM Fernando,31.08,1,11,...,81.00,37.1,0.0749,78.8,15.6,4.95,High,37.378593,0.919920,High
1,20210129_GalleInternational_Galle_AMFernando,2021-01-29,2021-01-31,3,Galle,Galle International Stadium,AM Fernando,31.08,2,33,...,88.70,81.0,0.0973,71.5,31.9,5.87,High,43.094260,0.917269,High
2,20210129_GalleInternational_Galle_AMFernando,2021-01-29,2021-02-01,4,Galle,Galle International Stadium,AM Fernando,31.09,1,12,...,87.20,51.4,0.0629,76.0,18.5,5.97,High,30.258701,0.899922,High
3,20210129_GalleInternational_Galle_AMFernando,2021-01-29,2021-02-02,5,Galle,Galle International Stadium,AM Fernando,31.09,1,22,...,206.90,99.0,0.1088,69.6,52.2,6.02,High,40.982648,0.942723,High
4,20210129_GalleInternational_Galle_AMFernando,2021-01-29,2021-02-02,5,Galle,Galle International Stadium,AM Fernando,31.09,2,16,...,259.15,69.6,0.1279,70.5,52.9,5.93,High,48.163862,0.939466,High


In [2]:

y_risk = df["injury_risk_label_v3"].astype(str)

y_fatigue = df["fatigue_score_0_100_v3"].astype(float)

y_risk.value_counts()


injury_risk_label_v3
Low       1992
Medium    1645
High      1363
Name: count, dtype: int64

In [3]:


feature_cols = [
    
    "deliveries_match",
    "deliveries_7d",
    "deliveries_28d",
    "acwr_std",
    "days_since_prev",
    
    
    "match_day",
    
    
    "avg_temp_c",
    "avg_humidity_pct",
    "precip_mm",
    "esi_norm",
    
    
    "age_years",
    
    
    "inferred_fielding_time_minutes",
]

X = df[feature_cols].copy()


X["days_since_prev"] = X["days_since_prev"].fillna(X["days_since_prev"].median())

X.describe().T


,count,mean,std,min,25%,50%,75%,max
deliveries_match,5000.0,25.309400,13.898874,6.000000,15.000000,24.000000,33.000000,90.000000
deliveries_7d,5000.0,134.395600,87.200850,6.000000,66.000000,121.000000,185.000000,630.000000
deliveries_28d,5000.0,384.751400,189.520159,6.000000,251.000000,376.500000,505.000000,1165.000000
acwr_std,5000.0,1.650099,1.116240,0.045600,0.836725,1.379500,2.110475,4.000000
days_since_prev,5000.0,2.169400,6.296335,0.000000,0.000000,0.000000,1.000000,63.000000
match_day,5000.0,3.023600,1.417549,1.000000,2.000000,3.000000,4.000000,5.000000
avg_temp_c,5000.0,27.045535,1.875812,22.000000,26.537519,27.459824,28.262213,31.145782
avg_humidity_pct,5000.0,81.166824,4.511244,67.667791,77.973138,81.157423,84.340090,95.000000
precip_mm,5000.0,3.579858,6.811266,0.000000,0.000000,1.361310,3.830868,54.270202
esi_norm,5000.0,0.401611,0.095685,0.051400,0.372000,0.424450,0.463500,0.622300


In [4]:

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_risk_enc = le.fit_transform(y_risk) 
list(le.classes_)


['High', 'Low', 'Medium']

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/raw/bowlerguard_5000_v3.csv")

target_cls = "injury_risk_label_v3"
target_reg = "fatigue_score_0_100_v3"


label_map = {"Low": 0, "Medium": 1, "High": 2}

df["y_risk"] = df[target_cls].map(label_map).astype(int)

df["y_fatigue"] = df[target_reg].astype(float)


feature_cols = [
    "deliveries_match", "deliveries_7d", "deliveries_28d",
    "acwr_std", "days_since_prev", "match_day",
    "avg_temp_c", "avg_humidity_pct", "precip_mm", "esi_norm",
    "age_years", "inferred_fielding_time_minutes"
]


feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y_cls = df["y_risk"]
y_reg = df["y_fatigue"]


if "match_date" in df.columns:
    print("Performing Temporal Split based on 'match_date'...")
    df_sorted = df.sort_values("match_date")
    
    
    cut = int(len(df_sorted) * 0.8)
    
    train_idx = df_sorted.index[:cut]
    test_idx = df_sorted.index[cut:]

    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]
    
    y_train_cls = y_cls.loc[train_idx]
    y_test_cls = y_cls.loc[test_idx]
    
    y_train_reg = y_reg.loc[train_idx]
    y_test_reg = y_reg.loc[test_idx]
    
else:
    print("Performing Stratified Random Split...")
   
    X_train, X_test, y_train_cls, y_test_cls = train_test_split(
        X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
    )
    
    
    _, _, y_train_reg, y_test_reg = train_test_split(
        X, y_reg, test_size=0.2, random_state=42, stratify=y_cls
    )


print("-" * 30)
print(f"Train Shape: {X_train.shape}")
print(f"Test Shape:  {X_test.shape}")
print("-" * 30)
print("Train Label Distribution (Risk):")
print(y_train_cls.value_counts(normalize=True))
print("-" * 30)
print("Test Label Distribution (Risk):")
print(y_test_cls.value_counts(normalize=True))
print("-" * 30)

Performing Temporal Split based on 'match_date'...
------------------------------
Train Shape: (4000, 12)
Test Shape:  (1000, 12)
------------------------------
Train Label Distribution (Risk):
y_risk
0    0.39850
1    0.33075
2    0.27075
Name: proportion, dtype: float64
------------------------------
Test Label Distribution (Risk):
y_risk
0    0.398
1    0.322
2    0.280
Name: proportion, dtype: float64
------------------------------


In [6]:

from pathlib import Path

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)


X_train.to_parquet(OUT/"X_train.parquet", index=True)
X_test.to_parquet(OUT/"X_test.parquet", index=True)


pd.Series(y_train_cls, index=X_train.index).to_csv(OUT/"y_train_risk.csv")
pd.Series(y_test_cls, index=X_test.index).to_csv(OUT/"y_test_risk.csv")


pd.Series(y_train_reg, index=X_train.index).to_csv(OUT/"y_train_fatigue.csv")
pd.Series(y_test_reg, index=X_test.index).to_csv(OUT/"y_test_fatigue.csv")

label_classes = ["Low", "Medium", "High"]
pd.Series(label_classes).to_csv(OUT/"risk_label_classes.csv", index=False, header=False)

print("Saved feature splits + label classes.")



Saved feature splits + label classes.


In [7]:
print("Total rows:", len(df))
print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

print("\nTrain class distribution:")
print(y_train_cls.value_counts())
print("\nTest class distribution:")
print(y_test_cls.value_counts())

Total rows: 5000
Train rows: 4000
Test rows: 1000

Train class distribution:
y_risk
0    1594
1    1323
2    1083
Name: count, dtype: int64

Test class distribution:
y_risk
0    398
1    322
2    280
Name: count, dtype: int64
